# SentinelAI: Pretrained ECAPA-TDNN Speaker Embedding & Verification Demo

**Project:** SentinelAI Voice Verification Research Module  
**Model:** Pretrained ECAPA-TDNN (`speechbrain/spkrec-ecapa-voxceleb`)  
**Objective:** Demonstrate end-to-end audio preprocessing, speaker embedding extraction, and cosine similarity comparison without requiring any public speaker datasets (such as VoxCeleb).

---

### Important Notes
1. **Pretrained Inference:** This notebook utilizes a pretrained ECAPA-TDNN model trained by SpeechBrain on VoxCeleb 1 & 2. The model is reused strictly for inference and was not trained from scratch by this project.
2. **No Dataset Downloads:** No large public datasets are needed. You can test with dynamically generated synthetic signals or small consented human voice recordings.
3. **Hardware Compatibility:** Supports both GPU (CUDA) and CPU execution. On Google Colab, a free T4 GPU can be used; on laptops without NVIDIA GPUs, it runs seamlessly on CPU.

## Step 1: Install Dependencies

Install PyTorch, SpeechBrain, Librosa, SoundFile, and SentinelAI dependencies.

In [ ]:
# Install dependencies in Google Colab
!pip install --quiet torch torchaudio speechbrain librosa soundfile pydantic fastapi

## Step 2: Import Modules & Check Compute Device

In [ ]:
import torch
import numpy as np
import soundfile as sf
from pathlib import Path

# Check hardware device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[SentinelAI] Active Compute Device: {device.upper()}")
if device == "cuda":
    print(f"[SentinelAI] GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("[SentinelAI] Running on CPU (compatible with low-spec development laptops).")

## Step 3: Initialize Preprocessor and ECAPA-TDNN Model Loader

We initialize:
- `AudioPreprocessor`: Enforces 16,000 Hz sample rate, mono downmixing, duration checks, and peak normalization.
- `ECAPAModelLoader`: Loads and caches the pretrained `speechbrain/spkrec-ecapa-voxceleb` model.

In [ ]:
from sentinelai.audio import AudioPreprocessor, AudioPreprocessingConfig
from sentinelai.speaker import (
    SpeakerConfig,
    ECAPAModelLoader,
    SpeakerEmbeddingExtractor,
    compute_cosine_similarity,
)

# Configure audio preprocessor for ECAPA-TDNN standards (16 kHz mono)
audio_cfg = AudioPreprocessingConfig(
    target_sample_rate=16000,
    min_duration_sec=0.5,
    max_duration_sec=30.0,
    normalize_audio=True,
)
preprocessor = AudioPreprocessor(config=audio_cfg)

# Configure speaker module
speaker_cfg = SpeakerConfig(
    model_source="speechbrain/spkrec-ecapa-voxceleb",
    device=device,
    normalize_embeddings=True,
)
model_loader = ECAPAModelLoader(config=speaker_cfg)
extractor = SpeakerEmbeddingExtractor(config=speaker_cfg, loader=model_loader, preprocessor=preprocessor)

print("[SentinelAI] Preprocessor and Extractor initialized successfully.")

## Step 4: Generate Synthetic Test Signals (Pipeline Verification)

To verify the pipeline before testing with consented voice recordings, we generate two deterministic mathematical signals (44.1 kHz stereo sine waves).

*Notice: These synthetic signals are purely mathematical test fixtures used to verify the code paths, shapes, and normalization without dataset downloads.*

In [ ]:
output_dir = Path("demo_audio")
output_dir.mkdir(exist_ok=True)

# Signal 1: 440 Hz tone (2.0s, 44.1 kHz stereo)
sr = 44100
t = np.linspace(0.0, 2.0, sr * 2, endpoint=False)
sig1 = 0.5 * np.sin(2.0 * np.pi * 440.0 * t)
sig1_stereo = np.column_stack([sig1, sig1]).astype(np.float32)
wav_path1 = output_dir / "synthetic_signal_1.wav"
sf.write(str(wav_path1), sig1_stereo, sr, subtype="PCM_16")

# Signal 2: 880 Hz tone (2.0s, 44.1 kHz stereo)
sig2 = 0.4 * np.sin(2.0 * np.pi * 880.0 * t)
sig2_stereo = np.column_stack([sig2, sig2]).astype(np.float32)
wav_path2 = output_dir / "synthetic_signal_2.wav"
sf.write(str(wav_path2), sig2_stereo, sr, subtype="PCM_16")

print(f"Generated: {wav_path1} and {wav_path2}")

## Step 5: Preprocessing & Speaker Embedding Extraction

We pass the WAV files to the extractor. The extractor runs the preprocessing pipeline (downmixing to mono, resampling to 16 kHz) and feeds the result through the pretrained ECAPA-TDNN network.

In [ ]:
# Extract embeddings
print("[SentinelAI] Extracting embedding for Signal 1...")
result1 = extractor.extract(wav_path1, source_id="signal-1")

print("[SentinelAI] Extracting embedding for Signal 2...")
result2 = extractor.extract(wav_path2, source_id="signal-2")

print("\n--- Speaker Embedding Results ---")
print(f"Model ID          : {result1.model_id}")
print(f"Inference Device  : {result1.device}")
print(f"Embedding Dimension: {result1.dimension} (dynamically verified)")
print(f"L2 Normalized     : {result1.normalized} (L2 Norm: {np.linalg.norm(result1.embedding):.4f})")
print(f"Timestamp (UTC)   : {result1.extraction_timestamp}")
print(f"Vector Preview    : {result1.embedding[:6]}...")

## Step 6: Cosine Similarity Comparison

Cosine similarity measures the directional alignment between two voiceprint embeddings:

$$\text{similarity} = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

- **+1.0**: Perfect directional alignment (identical embedding).
- **~0.0**: Orthogonal (independent / distinct acoustic profiles).
- **-1.0**: Opposite directional alignment.

In [ ]:
# Self-similarity of Signal 1
self_sim = compute_cosine_similarity(result1, result1)
print(f"Self-Similarity (Signal 1 vs Signal 1): {self_sim:.6f} (Expected: 1.000000)")

# Cross-similarity between Signal 1 and Signal 2
cross_sim = compute_cosine_similarity(result1, result2)
print(f"Cross-Similarity (Signal 1 vs Signal 2): {cross_sim:.6f}")

## Step 7: Testing with Consented Voice Recordings

To test with real speech recordings from consented human speakers:
1. Record two short utterances (1 to 5 seconds each) speaking into a microphone.
2. Upload them below (or place them in your working folder).
3. Run the cell to compare the embeddings.

In [ ]:
# In Google Colab, you can upload files interactively:
# from google.colab import files
# uploaded = files.upload()

# Example comparison function
def compare_voice_recordings(file_path_a: str, file_path_b: str) -> None:
    emb_a = extractor.extract(file_path_a, source_id="voice-a")
    emb_b = extractor.extract(file_path_b, source_id="voice-b")
    score = compute_cosine_similarity(emb_a, emb_b)
    
    print(f"Similarity Score: {score:.4f}")
    print("Note: High similarity (> 0.70) typically indicates the same speaker in calibrated setups,")
    print("while lower similarity indicates distinct speakers. Calibrated thresholds will be introduced in Phase 3.")

## Summary and Next Steps

- **Phase 1 Complete:** Audio preprocessing (mono downmixing, 16 kHz resampling, duration & numeric validation).
- **Phase 2 Complete:** Pretrained ECAPA-TDNN integration, 192-dim embedding extraction, L2 normalization, and cosine similarity.
- **Phase 3 (Upcoming):** Voice enrollment, multi-utterance averaging, threshold calibration, and authentication API.